# 02. NV2: Phân cụm và Nhận diện hành vi chiếm dụng trạm sạc

## Mục tiêu nghiên cứu (NV2)
Nghiên cứu các kiểu hành vi kết nối và chiếm dụng cổng sạc sau khi xe đã được nạp đầy:
1. Phân loại các nhóm phiên sạc dựa trên đặc trưng:
   - Thời gian sạc thực tế ($T_{charging}$)
   - Thời gian nhàn rỗi ($T_{idle} = T_{total} - T_{charging}$)
   - Tỷ lệ chiếm dụng nhàn rỗi ($Idle\ Ratio = T_{idle} / T_{total}$)
   - Năng lượng tiêu thụ ($E$)
2. Áp dụng các thuật toán phân cụm không giám sát: **K-Means**, **Gaussian Mixture Models (GMM)** hoặc **DBSCAN**.
3. Định danh và xây dựng hồ sơ hành vi người dùng (vd: Nhóm sạc nhanh rời đi ngay, Nhóm chiếm dụng giờ làm việc, Nhóm sạc qua đêm, v.v.).


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
from src.config import SESSIONS_FEATURES_PARQUET, FIGURES_DIR
from src.visualization import set_plotting_style

set_plotting_style()

# Tải dữ liệu đặc trưng đã tiền xử lý
df = pd.read_parquet(SESSIONS_FEATURES_PARQUET)
print(f"Kích thước tập dữ liệu đặc trưng: {df.shape[0]:,} dòng, {df.shape[1]} cột")


### 1. Khảo sát phân bố thời gian nhàn rỗi ($T_{idle}$) và $Idle\ Ratio$


In [ ]:
# Lọc các phiên hợp lệ
valid = df[df['is_valid_session']].copy()

print("Thống kê mô tả hành vi chiếm dụng:")
print(valid[['total_duration_hours', 'charging_time_hours', 'idle_time_hours', 'idle_ratio']].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(valid['idle_time_hours'], bins=50, ax=axes[0], color='#d62728', kde=True)
axes[0].set_title('Phân phối Thời gian nhàn rỗi (T_idle)')
axes[0].set_xlabel('Thời gian nhàn rỗi (giờ)')
axes[0].set_xlim(0, 15)

sns.histplot(valid['idle_ratio'], bins=50, ax=axes[1], color='#9467bd', kde=True)
axes[1].set_title('Phân phối Tỷ lệ nhàn rỗi (Idle Ratio)')
axes[1].set_xlabel('Idle Ratio')
plt.tight_layout()
plt.show()


### 2. Chuẩn hóa đặc trưng & Lựa chọn số cụm tối ưu (Elbow Method & Silhouette Score)


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

features_for_clustering = ['charging_time_hours', 'idle_time_hours', 'idle_ratio', 'Energy (kWh)']
X = valid[features_for_clustering].dropna()

# Lấy mẫu ngẫu nhiên đại diện để xác định số cụm nhanh
sample_X = X.sample(n=min(30000, len(X)), random_state=42)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(sample_X)

inertias = []
k_range = range(2, 8)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(k_range, inertias, 'bo-')
plt.title('Phương pháp đường gấp khúc (Elbow Method)')
plt.xlabel('Số cụm k')
plt.ylabel('Inertia')
plt.show()


### 3. Huấn luyện mô hình phân cụm K-Means & Định danh hồ sơ cụm


In [ ]:
# Huấn luyện K-Means với k tối ưu (ví dụ k=4)
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
sample_X['cluster'] = kmeans.fit_predict(X_scaled)

cluster_profile = sample_X.groupby('cluster')[features_for_clustering].mean()
cluster_profile['session_count'] = sample_X['cluster'].value_counts()
display(cluster_profile)
